# harness_v2 (M1) trên Colab

Chạy agent tự viết trên `unseen_50` / `private_50`, in PA. Cần GPU (A100/G4). Cần Colab Secret `HF_TOKEN` đọc được dataset repo private `Crinkk/datamate-harness-v2` (đổi bằng env `HV2_REPO`).

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path
def sh(cmd): print("$", cmd); subprocess.run(cmd, shell=True, check=True)
HV2_REPO = os.environ.get("HV2_REPO", "Crinkk/datamate-harness-v2")   # dataset repo private chứa code.zip + data.zip
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception: pass
sh("pip install -q -U huggingface_hub")
from huggingface_hub import hf_hub_download
import shutil
PROJ = Path("/content/proj")
if os.environ.get("HV2_REFRESH_DATA", "0") == "1" or not (PROJ / "data").exists():   # data.zip lớn: chỉ tải lần đầu
    shutil.rmtree(PROJ, ignore_errors=True)
    zipfile.ZipFile(hf_hub_download(HV2_REPO, "harness_v2_data.zip", repo_type="dataset")).extractall(PROJ)
shutil.rmtree(PROJ / "harness_v2", ignore_errors=True)   # code.zip nhỏ: luôn lấy bản mới nhất
zipfile.ZipFile(hf_hub_download(HV2_REPO, "harness_v2_code.zip", repo_type="dataset", force_download=True)).extractall(PROJ)
sh("pip install -q pandas openpyxl rapidfuzz")
from importlib.metadata import version
try: _v = tuple(int(x) for x in version("vllm").split(".")[:3])
except Exception: _v = (0,)
if _v < (0, 19, 1): sh("pip install -q -U 'vllm>=0.19.1'")
import torch; assert torch.cuda.is_available(), "Cần GPU"
print("OK", PROJ)

In [ ]:
import os, subprocess, time, urllib.request
from pathlib import Path
MODEL_ID = os.environ.get("HV2_MODEL_ID", "ornith-ai/Ornith-1.5-9B")
PORT = 8000
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception: pass
PARSER = os.environ.get("HV2_TOOL_PARSER", "qwen3_xml")           # parser gọi tool của vLLM; đổi theo họ model
REASONING = os.environ.get("HV2_REASONING_PARSER", "qwen3")        # "none" nếu model không có chế độ suy nghĩ
TEMPLATE = os.environ.get("HV2_CHAT_TEMPLATE", "/content/proj/harness_v2/ollama_template.jinja")   # "none" = dùng template có sẵn của model
cmd = ["vllm", "serve", MODEL_ID, "--served-model-name", "ornith", "--host", "127.0.0.1", "--port", str(PORT),
       "--max-model-len", os.environ.get("HV2_MAX_LEN", "65536"), "--gpu-memory-utilization", "0.90", "--max-num-seqs", "8", "--enable-prefix-caching",
       "--enable-auto-tool-choice", "--tool-call-parser", PARSER, "--trust-remote-code", "--seed", "0"]
if REASONING.lower() != "none": cmd += ["--reasoning-parser", REASONING]
if TEMPLATE.lower() != "none": cmd += ["--chat-template", TEMPLATE]
if os.environ.get("HV2_TEXT_ONLY", "1") == "1": cmd += ["--limit-mm-per-prompt", '{"image": 0, "video": 0}']   # model không có phần ảnh thì đặt 0
log = open("/content/vllm_server.log", "w")
SERVER = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while time.time() - t0 < 1800:
    if SERVER.poll() is not None: raise RuntimeError("vLLM thoát sớm:\n" + "".join(open("/content/vllm_server.log").readlines()[-40:]))
    try:
        if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2).status == 200: break
    except Exception: time.sleep(3)
else: raise RuntimeError("vLLM không lên sau 30 phút")
print("vLLM sẵn sàng sau", round(time.time() - t0), "s")

In [ ]:
# Cấu hình lần chạy (đổi rồi chạy lại cell này; mỗi RUN_TAG ghi vào thư mục riêng)
SET     = os.environ.get("HV2_SET", "private_50")      # unseen_50 | private_50
RUN_TAG = os.environ.get("HV2_TAG", "agent_v1")
WORKERS = int(os.environ.get("HV2_WORKERS", 4))
LIMIT   = int(os.environ.get("HV2_LIMIT", 0)) or None  # đặt 5 để chạy thử vài câu
os.environ.update({"HV2_BASE_URL": f"http://127.0.0.1:{PORT}/v1", "HV2_MODEL": "ornith",
                   "HV2_TRACE_DIR": f"/content/proj/runs/{RUN_TAG}_{SET}/traces"})
out = f"runs/{RUN_TAG}_{SET}/predictions.jsonl"
cmd = f"cd /content/proj && python -m harness_v2.run --set {SET} --solver harness_v2.solvers.agent_v1:solve --out {out} --workers {WORKERS}" + (f" --limit {LIMIT}" if LIMIT else "")
get_ipython().system(cmd)

In [ ]:
get_ipython().system(f"cd /content/proj && python -m harness_v2.score --set {SET} --preds {out}")
import shutil; shutil.make_archive(f"/content/{RUN_TAG}_{SET}", "zip", f"/content/proj/runs/{RUN_TAG}_{SET}")
print(f"Kết quả + trace: /content/{RUN_TAG}_{SET}.zip (tải về từ panel Files)")